# Shallow and in-place migration

Reuse compatible Parquet files, inspect the metadata that Iceberg adds, and prove that migration did not copy the original files.

![Shallow and in-place migration](assets/02_flow.png)


## Connect the notebook to Spark

Run from top to bottom in a fresh Python kernel. Close the previous Spark session first.
Connection values use the environment when present, otherwise the Kubernetes Service names below.
Credentials stay in the existing course environment. No extra workbench manifest update is needed.
Short Service names resolve within the Pod's namespace; use a fully qualified Service name when crossing namespaces.
A Service name stays stable when its Pods are replaced. A container name or Pod IP is not the connection contract.

The notebook kernel is the **driver**. `.master(master_url)` connects this application to the
standalone Spark master. The master allocates executors on its workers. Workers must also reach
the notebook driver address and its ports. The Spark services and workbench run inside Kubernetes;
this is a standalone Spark application, rather than native Kubernetes submission.

No separate `spark-submit` command is needed while running notebook cells. Read the master URL
from the environment and inspect `spark.sparkContext.master` after the connection is created.


In [ ]:
# Import the standard Python and Spark classes.
import os
import re
import socket
import uuid

from datetime import date, datetime
from decimal import Decimal

from pyspark.sql import SparkSession


In [ ]:
# Use environment overrides or the stable Kubernetes Service names.
master_url = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
driver_host = os.environ.get('SPARK_DRIVER_HOST', 'jupyter')

hdfs_uri = os.environ.get('HDFS_URI', 'hdfs://namenode:9000').rstrip('/')
metastore_uri = os.environ.get('HIVE_METASTORE_URI', 'thrift://hive-metastore:9083')

polaris_base_url = os.environ.get('POLARIS_BASE_URL', 'http://polaris:8181').rstrip('/')
polaris_uri = os.environ.get('POLARIS_URI', polaris_base_url + '/api/catalog').rstrip('/')
polaris_warehouse = os.environ.get('POLARIS_WAREHOUSE', os.environ.get('POLARIS_CATALOG', 'training'))
s3_endpoint = os.environ.get('S3_ENDPOINT', 'http://seaweedfs:8333')
storage_region = os.environ.get('AWS_DEFAULT_REGION', os.environ.get('AWS_REGION', 'us-east-1'))
polaris_realm = os.environ.get('POLARIS_REALM', 'POLARIS')

assert master_url.startswith('spark://'), 'Use the course standalone Spark master URL'


In [ ]:
# Display the resolved connection values before Spark starts.
resolved_connections = {
    'Spark master': master_url,
    'Notebook driver': driver_host,
    'HDFS filesystem': hdfs_uri,
    'Hive metastore': metastore_uri,
    'Polaris REST catalog': polaris_uri,
    'Polaris catalog name': polaris_warehouse,
    'Polaris realm': polaris_realm,
    'S3 endpoint': s3_endpoint,
    'Storage region': storage_region,
}

for setting, value in resolved_connections.items():
    print(f'{setting:24} = {value}')


In [ ]:
# Read existing credentials without printing their values.
polaris_credential = os.environ.get('POLARIS_CREDENTIAL')
if not polaris_credential:
    polaris_credential = (
        os.environ['POLARIS_CLIENT_ID'] + ':' + os.environ['POLARIS_CLIENT_SECRET']
    )

s3_access_key = os.environ['AWS_ACCESS_KEY_ID']
s3_secret_key = os.environ['AWS_SECRET_ACCESS_KEY']


### Give this execution its own tables and files

`ns` is the database or namespace used by the table names below. `root` is this execution's HDFS directory.
A new run ID keeps a full rerun separate from earlier work. Keep the printed values if you want to inspect the results later.


In [ ]:
# Choose isolated names for this notebook execution.
run_id = os.environ.get('LESSON_RUN_ID', uuid.uuid4().hex[:10])
assert re.fullmatch(r'[a-z0-9_]{1,32}', run_id), 'Use a lowercase execution ID'

ns = f'migration11_02_{run_id}'
root = hdfs_uri + f'/workshop/11-iceberg-migration/02/{run_id}'

print('Namespace:', ns)
print('HDFS directory:', root)


In [ ]:
# Make the warehouse locations and Polaris catalog selection explicit.
hive_warehouse = root + '/managed'
hadoop_iceberg_warehouse = root + '/iceberg-warehouse'
hive_iceberg_warehouse = root + '/iceberg-hive-warehouse'

print('Hive managed warehouse:      ', hive_warehouse)
print('HadoopCatalog warehouse:     ', hadoop_iceberg_warehouse)
print('Hive Iceberg warehouse:      ', hive_iceberg_warehouse)
print('Polaris warehouse parameter: ', polaris_warehouse, '(catalog name, not an S3 path)')


### Configure the driver and executors

`.master(master_url)` selects the cluster. The driver address is different: executors use it to call back to this notebook.
Driver ports must be reachable from the worker network. A small executor allocation is sufficient for these datasets.


In [ ]:
# Connect the application to the master and advertise the notebook driver.
builder = (
    SparkSession.builder
    .appName('iceberg-migration-02')
    .master(master_url)
    .config('spark.driver.host', driver_host)
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.driver.port', '7078')
    .config('spark.blockManager.port', '7079')
)


In [ ]:
# Set the resource limits, time zone and credential redaction.
builder = (
    builder
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.adaptive.enabled', 'false')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.openlineage.disabled', 'true')
    .config('spark.redaction.regex', '(?i)secret|password|token|access[.]?key|credential')
)


### Configure the catalogs

`spark_catalog` reads Hive tables and supports in-place conversion through the metastore.
`lake` is a HadoopCatalog: its catalog metadata and data live on HDFS, without the Hive metastore.
`iceberg` is a HiveCatalog: table registration lives in the Hive metastore, with files on HDFS.
`polaris` is a REST catalog for Iceberg tables in the object-store warehouse.
Catalog configuration is required before creating the Spark session.


In [ ]:
# Configure the default HDFS filesystem and managed-table warehouse.
builder = (
    builder
    .config('spark.hadoop.fs.defaultFS', hdfs_uri)
    .config('spark.hadoop.dfs.client.use.datanode.hostname', 'true')
    .config('spark.sql.warehouse.dir', hive_warehouse)
)


In [ ]:
# Enable Iceberg SQL and keep the Hive metastore available.
builder = (
    builder
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.hadoop.hive.metastore.uris', metastore_uri)
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
    .config('spark.sql.catalog.spark_catalog.type', 'hive')
    .config('spark.sql.catalog.spark_catalog.uri', metastore_uri)
    .config('spark.sql.catalog.spark_catalog.warehouse', hive_warehouse)
)


In [ ]:
# Keep the HDFS Iceberg warehouse under this execution directory.
builder = (
    builder
    .config('spark.sql.catalog.lake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lake.type', 'hadoop')
    .config('spark.sql.catalog.lake.warehouse', hadoop_iceberg_warehouse)
)


In [ ]:
# Register a Hive-metastore Iceberg catalog with HDFS storage.
builder = (
    builder
    .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.iceberg.type', 'hive')
    .config('spark.sql.catalog.iceberg.uri', metastore_uri)
    .config('spark.sql.catalog.iceberg.warehouse', hive_iceberg_warehouse)
)


### Configure both S3 clients

Spark reads and writes ordinary `s3a://` files through Hadoop S3A.
Iceberg's Polaris tables use `s3://` locations through S3FileIO. These are separate clients;
configure the endpoint, credentials and path-style access for both.
HTTP disables S3A TLS only for this course endpoint. HTTPS keeps TLS enabled.


In [ ]:
# Configure Hadoop S3A for ordinary Spark object-store files.
builder = (
    builder
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint', s3_endpoint)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', str(s3_endpoint.startswith('https://')).lower())
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.access.key', s3_access_key)
    .config('spark.hadoop.fs.s3a.secret.key', s3_secret_key)
)


In [ ]:
# Connect the Polaris REST catalog using credentials from the environment.
builder = (
    builder
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', polaris_uri)
    .config('spark.sql.catalog.polaris.warehouse', polaris_warehouse)
    .config('spark.sql.catalog.polaris.oauth2-server-uri', polaris_uri + '/v1/oauth/tokens')
    .config('spark.sql.catalog.polaris.credential', polaris_credential)
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.header.Polaris-Realm', polaris_realm)
    .config('spark.sql.catalog.polaris.rest-metrics-reporting-enabled', 'false')
)


In [ ]:
# Let Iceberg read and write the configured S3 warehouse.
builder = (
    builder
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', s3_endpoint)
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', s3_access_key)
    .config('spark.sql.catalog.polaris.s3.secret-access-key', s3_secret_key)
    .config('spark.sql.catalog.polaris.client.region', storage_region)
)


In [ ]:
# Start Spark and display the master actually used.
spark = builder.enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print('Spark version:', spark.version)
print('Connected master:', spark.sparkContext.master)

assert spark.sparkContext.master == master_url, 'Spark must use the configured master'


### Verify that work reaches an executor

The following small task runs on a worker. Its process hostname must differ from the notebook driver's hostname.
This checks that the application is using a separate executor instead of silently running in local mode.


In [ ]:
# Verify distributed execution before writing course data.
worker_names = (
    spark.sparkContext
    .parallelize([1], 1)
    .map(lambda _: __import__('socket').gethostname())
    .collect()
)

assert all(name != socket.gethostname() for name in worker_names), 'Use a separate executor'
print('PASS: Distributed Spark executor verified')


In [ ]:
# Put managed Hive data under this execution directory.
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS spark_catalog.{ns}
    LOCATION '{root}/managed'
""")


In [ ]:
# Create the namespace for HDFS Iceberg exercises.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS lake.{ns}
""")


In [ ]:
# Use the existing Hive database through the named Iceberg catalog.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS iceberg.{ns}
""")


In [ ]:
# Create the namespace for object-store Iceberg exercises.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS polaris.{ns}
""")


## Name the operation precisely
In this lesson, shallow migration means adding existing files to a new Iceberg table without rewriting them.
In-place migration converts an existing Hive table registration to Iceberg and creates a backup registration.
Both approaches create Iceberg metadata. A pointer change alone cannot convert a directory of Parquet files.
`register_table` registers existing Iceberg metadata; it does not create that metadata from Hive data.

Stop source writers before either ownership transfer. Imported files become owned by Iceberg and can later be
deleted by Iceberg maintenance. Do not keep an independent Hive writer or purge shared source files.
Prefer `migrate` or `snapshot` when appropriate; `add_files` needs careful schema and ownership checks.


In [ ]:
# Define the staged rows and their explicit column types.
staged_invoices_df = spark.createDataFrame(
    [
        (501, 101, Decimal('120.00'), date(2025, 1, 1)),
        (502, 102, Decimal('80.00'), date(2025, 1, 1)),
        (503, 103, Decimal('200.00'), date(2025, 1, 2)),
    ],
    """
        invoice_id INT,
        order_id INT,
        amount DECIMAL(12,2),
        invoice_date DATE
    """,
)


In [ ]:
# Choose the invoice source directory.
invoice_path = root + '/invoices/unpartitioned'

staged_invoices_df.write.mode('errorifexists').parquet(invoice_path)


In [ ]:
# Make these rows available to Spark SQL.
staged_invoices_df.createOrReplaceTempView('staged_invoices')


In [ ]:
# Create the table for this step.
spark.sql(f"""
    CREATE TABLE spark_catalog.{ns}.invoices_source
    USING parquet
    LOCATION '{invoice_path}'
""")


In [ ]:
# Keep the original paths so file reuse can be verified.
original_files = set(spark.table(f'spark_catalog.{ns}.invoices_source').inputFiles())


In [ ]:
# Create the table for this step.
spark.sql(f"""
    CREATE TABLE lake.{ns}.invoices (
    invoice_id INT, order_id INT, amount DECIMAL(12,2), invoice_date DATE
    )
    USING iceberg
    TBLPROPERTIES ('format-version'='2')
""")


## Shallow import
Check every source column and its physical type before import. `add_files` does not establish schema compatibility for you.
Our source has an explicit schema and a full read-and-compare check before ownership transfers.


In [ ]:
# Invoice source schema matches.
assert spark.table(f'spark_catalog.{ns}.invoices_source').schema == staged_invoices_df.schema, 'Invoice source schema matches'

print('PASS: Invoice source schema matches')


In [ ]:
# No unmatched rows should remain in this direction.
assert spark.sql(f"""
    (SELECT *
    FROM staged_invoices) EXCEPT ALL (SELECT *
    FROM spark_catalog.{ns}.invoices_source)
""").count() == 0, 'No unmatched rows should remain in this direction'


In [ ]:
# No unmatched rows should remain in this direction.
assert spark.sql(f"""
    (SELECT *
    FROM spark_catalog.{ns}.invoices_source) EXCEPT ALL (SELECT *
    FROM staged_invoices)
""").count() == 0, 'No unmatched rows should remain in this direction'

print('PASS: complete row reconciliation')


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL lake.system.add_files(
    table => '{ns}.invoices', source_table => '{ns}.invoices_source', check_duplicate_files => true
    )
""").show(truncate=False)


In [ ]:
# Import reused the exact original file paths.
assert {file_row.file_path for file_row in spark.sql(f"""
    SELECT file_path
    FROM lake.{ns}.invoices.files
""").collect()} == original_files, 'Import reused the exact original file paths'

print('PASS: Import reused the exact original file paths')


In [ ]:
# No unmatched rows should remain in this direction.
assert spark.sql(f"""
    (SELECT *
    FROM staged_invoices) EXCEPT ALL (SELECT *
    FROM lake.{ns}.invoices)
""").count() == 0, 'No unmatched rows should remain in this direction'


In [ ]:
# No unmatched rows should remain in this direction.
assert spark.sql(f"""
    (SELECT *
    FROM lake.{ns}.invoices) EXCEPT ALL (SELECT *
    FROM staged_invoices)
""").count() == 0, 'No unmatched rows should remain in this direction'

print('PASS: complete row reconciliation')


In [ ]:
# Remove only the specified catalog object.
spark.sql(f"""
    DROP TABLE spark_catalog.{ns}.invoices_source
""")


In [ ]:
# Iceberg owns and reads the imported files.
assert spark.table(f'lake.{ns}.invoices').count() == 3, 'Iceberg owns and reads the imported files'

print('PASS: Iceberg owns and reads the imported files')


## In-place conversion of a partitioned table
Use a different dataset so the two migration approaches never claim the same files.
The backup name preserves the original registration, not an independent copy of its data.
The source partition specification remains an identity specification; this operation does not automatically introduce day transforms.


In [ ]:
# Define the staged rows and their explicit column types.
staged_order_items_df = spark.createDataFrame(
    [
        (101, 1, 'PEN', 2, date(2025, 1, 1)),
        (101, 2, 'BOOK', 1, date(2025, 1, 1)),
        (103, 1, 'BAG', 1, date(2025, 1, 2)),
    ],
    """
        order_id INT,
        line_no INT,
        product STRING,
        quantity INT,
        order_date DATE
    """,
)


In [ ]:
# Choose the order-item source directory.
items_path = root + '/order-items/partitioned'

staged_order_items_df.write.mode('errorifexists').partitionBy('order_date').parquet(items_path)


In [ ]:
# Make these rows available to Spark SQL.
staged_order_items_df.createOrReplaceTempView('staged_order_items')


In [ ]:
# Register the external source files.
spark.sql(f"""
    CREATE EXTERNAL TABLE spark_catalog.{ns}.order_items (
    order_id INT, line_no INT, product STRING, quantity INT
    )
    PARTITIONED BY (order_date DATE)
    STORED AS PARQUET
    LOCATION '{items_path}'
""")


In [ ]:
# Register the source partition directories.
spark.sql(f"""
    MSCK REPAIR TABLE spark_catalog.{ns}.order_items
""")


In [ ]:
# Keep the order-item file paths before conversion.
items_files = set(spark.table(f'spark_catalog.{ns}.order_items').inputFiles())


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL spark_catalog.system.migrate(
    table => '{ns}.order_items', properties => map('format-version', '2')
    )
""").show(truncate=False)


In [ ]:
# In-place migration reused all original files.
assert {file_row.file_path for file_row in spark.sql(f"""
    SELECT file_path
    FROM spark_catalog.{ns}.order_items.files
""").collect()} == items_files, 'In-place migration reused all original files'

print('PASS: In-place migration reused all original files')


In [ ]:
# No unmatched rows should remain in this direction.
assert spark.sql(f"""
    (SELECT *
    FROM staged_order_items) EXCEPT ALL (SELECT *
    FROM spark_catalog.{ns}.order_items)
""").count() == 0, 'No unmatched rows should remain in this direction'


In [ ]:
# No unmatched rows should remain in this direction.
assert spark.sql(f"""
    (SELECT *
    FROM spark_catalog.{ns}.order_items) EXCEPT ALL (SELECT *
    FROM staged_order_items)
""").count() == 0, 'No unmatched rows should remain in this direction'

print('PASS: complete row reconciliation')


In [ ]:
# Inspect the catalog metadata.
spark.sql(f"""
    SHOW TABLES IN spark_catalog.{ns}
""").show(truncate=False)


In [ ]:
# Read the rows or metadata needed for this check.
spark.sql(f"""
    SELECT *
    FROM spark_catalog.{ns}.order_items.snapshots
""").show(truncate=False)


In [ ]:
# Read the rows or metadata needed for this check.
spark.sql(f"""
    SELECT partition, record_count, file_path
    FROM spark_catalog.{ns}.order_items.files
""").show(truncate=False)


In [ ]:
# Migration backup registration exists.
assert spark.catalog.tableExists(f'spark_catalog.{ns}.order_items_BACKUP_'), 'Migration backup registration exists'

print('PASS: Migration backup registration exists')


## Classroom exercises and assignment

1. Record which files changed and which metadata was added.
2. Explain why the backup registration must remain read-only.
3. Assignment: choose shallow import, in-place migration or rewrite for a source with incompatible decimal types. Document file ownership after cutover.


<details>
    <summary>Review points</summary>

    File paths remain the same for these two approaches. An incompatible physical schema needs normalization and a rewrite. The backup shares files: housekeeping on the new owner can invalidate it. A rollback requires a writer freeze and a defined treatment of post-cutover writes, not just a rename.

</details>


In [ ]:
# Display the checked result.
print('ALL CORE CHECKS PASSED')


In [ ]:
# Release this notebook Spark session.
spark.stop()
